# Bielik - agent ReAct z [DSPy](https://dspy.ai/)

[Bielik](https://bielik.ai/) to polski model językowy stworzony przez [SpeakLeash](https://speakleash.org/) i ICM. W tym notebooku zbudujemy agenta pogodowego (analogicznego do tego z `003-01. LLM-function_calling.ipynb`) opartego o model `SpeakLeash/bielik-11b-v3.0-instruct:bf16` hostowany lokalnie z użyciem [Ollama](https://ollama.com/).

[DSPy](https://dspy.ai/) (poznane w `009-01`) ma jeden gotowy moduł agentowy - `dspy.ReAct` - który zarządza pętlą Thought/Action/Observation, parsując wyjście modelu jako tekst. To jedyna ścieżka agentowa w DSPy: framework nie udostępnia osobnej klasy "function calling agent" (jak np. `FunctionAgent` w LlamaIndex), bo cała filozofia DSPy opiera się na komponowaniu prymitywów (`Predict`, `ChainOfThought`, `ReAct`) we własne moduły, a nie na dostarczaniu wielu wyspecjalizowanych klas agentów.

Pozostałe trzy notebooki w tej serii pokazują tę samą funkcjonalność w innych podejściach:
- `010-01. Bielik-openai.ipynb` - manualna pętla function callingu na surowym SDK
- `010-02. Bielik-pydantic-ai.ipynb` - function calling z PydanticAI
- `010-04. Bielik-llama-index.ipynb` - dwa warianty: ReAct i function calling z LlamaIndex

`dspy.ReAct`:
- działa z każdym modelem rozumiejącym instrukcje (w tym Bielikiem na Ollamie),
- z definicji wymusza sekwencyjność - jedna akcja na turę,
- automatycznie buduje opisy narzędzi z _sygnatur_ i docstringów funkcji Pythonowych.

Pod spodem DSPy używa LiteLLM jako warstwy abstrakcji nad różnymi providerami modeli.

## Wymagania

1. Zainstalowana Ollama: https://ollama.com/download
2. Pobrany model Bielika: `ollama pull SpeakLeash/bielik-11b-v3.0-instruct:bf16`
3. Zbudowany model `bielik-tools` z customowego Modelfile:
   ```bash
   ollama create bielik-tools -f "010-00. Bielik.Modelfile"
   ```
   Używamy `bielik-tools` zamiast oryginalnego modelu również tutaj - mimo że ReAct nie wymaga strukturyzowanych `tool_calls`, customowy Modelfile naprawia bug ze stop tokens (oryginał używa tokenów Llamy 3 zamiast ChatML, co psuje generację). Szczegóły w `010-00. Bielik.Modelfile.md`.
4. Działający serwis Ollama w tle (port `11434`).

> **Uwaga dot. DevContainera:** Notebook działa wewnątrz kontenera Dockera, więc `localhost` z perspektywy notebooka wskazuje na kontener, a nie hosta z uruchomioną Ollamą. W kodzie używamy `host.docker.internal`, aby z wnętrza kontenera dotrzeć do Ollamy działającej na hoście.

In [ ]:
import requests
import json

## Funkcja pobierająca współrzędne geograficzne dla danej nazwy

In [ ]:
def get_geolocation(location):
    """
    Pobiera współrzędne geograficzne oraz dane lokalizacyjne.

    Parametry:
    location (str): Nazwa lokalizacji, dla której chcemy uzyskać współrzędne geograficzne.

    Zwraca:
    dict: Dane lokalizacyjne w formacie JSON.
    """
    print(f"[tool_call] get_geolocation(location={location!r})")

    geocode_endpoint = "https://nominatim.openstreetmap.org/search"
    geocode_params = {"q": location, "format": "json"}
    headers = {"User-Agent": "Python script"}

    geocode_response = requests.get(geocode_endpoint, params=geocode_params, headers=headers)
    geocode_data = geocode_response.json()

    simplified_data = {
        "name": geocode_data[0]["display_name"],
        "latitude": geocode_data[0]["lat"],
        "longitude": geocode_data[0]["lon"]
    }
    return simplified_data

geolocation_data = get_geolocation("Poznań")
print(json.dumps(geolocation_data, indent=4, ensure_ascii=False))

## Funkcja pobierająca informacje o pogodzie dla podanych współrzędnych geograficznych

In [ ]:
def get_wind_direction(degrees):
    """Konwertuje kierunek wiatru ze stopni na nazwy kierunków świata."""
    directions = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
                  'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
    index = int((degrees + 11.25) // 22.5) % 16
    return directions[index]

def get_current_weather(latitude, longitude):
    """
    Pobiera aktualne dane pogodowe dla podanych współrzędnych geograficznych.

    Parametry:
    latitude (float): Szerokość geograficzna.
    longitude (float): Długość geograficzna.

    Zwraca:
    dict: Dane pogodowe w formacie JSON.
    """
    print(f"[tool_call] get_current_weather(latitude={latitude!r}, longitude={longitude!r})")

    weather_endpoint = "https://api.open-meteo.com/v1/forecast"
    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }

    weather_response = requests.get(weather_endpoint, params=weather_params)
    weather_data = weather_response.json()

    simplified_weather = {
        "temperature": f"{weather_data['current_weather']['temperature']} °C",
        "wind_speed": f"{weather_data['current_weather']['windspeed']} km/h",
        "wind_direction": get_wind_direction(weather_data['current_weather']['winddirection']),
        "is_day": "day" if weather_data['current_weather']['is_day'] else "night"
    }

    return simplified_weather

current_weather = get_current_weather(geolocation_data['latitude'], geolocation_data['longitude'])
print(json.dumps(current_weather, indent=4))

## Konfiguracja DSPy

DSPy używa biblioteki LiteLLM pod spodem, która ma natywne wsparcie dla Ollamy. Prefiks `ollama_chat/` mapuje zapytania na endpoint POST `/api/chat` Ollamy.

In [ ]:
import dspy

bielik_lm = dspy.LM(
    model="ollama_chat/bielik-tools",
    api_base="http://host.docker.internal:11434", # natywne API Ollamy - bez /v1
    temperature=0.3,
    max_tokens=2000,
    cache=False # wyłączenie cache, aby każde wywołanie naprawdę trafiło do modelu
)

# Ustawiamy Bielika jako domyślny LM dla modułów DSPy
dspy.configure(lm=bielik_lm)

## Sygnatura agenta i utworzenie modułu ReAct

W DSPy zamiast budować pętlę ręcznie, deklarujemy _sygnaturę_ (jak ma wyglądać wejście i wyjście) i przekazujemy listę narzędzi. `dspy.ReAct` sam zarządza pętlą Thought/Action/Observation.

Funkcje `get_geolocation` i `get_current_weather` trafiają jako narzędzia bez żadnej dodatkowej obróbki - DSPy odczyta ich nazwy, sygnatury argumentów i docstringi i automatycznie zbuduje opis dla modelu.

In [ ]:
class WeatherAgent(dspy.Signature):
    """Odpowiadaj użytkownikowi na pytania o pogodę. Jeśli do realizacji
    polecenia musisz ustalić współrzędne geograficzne jakiegoś miejsca,
    zawsze pobierz je za pomocą narzędzia get_geolocation - nigdy nie
    podawaj współrzędnych z własnej wiedzy. Nie każde polecenie wymaga
    ustalania współrzędnych."""

    user_question: str = dspy.InputField(desc="Pytanie użytkownika dotyczące pogody")
    answer: str = dspy.OutputField(desc="Odpowiedź dla użytkownika w języku polskim")


# dspy.ReAct sam zarządza pętlą Thought / Action / Observation.
# max_iters - twardy limit kroków agenta (zabezpieczenie przed zapętleniem).
weather_agent = dspy.ReAct(
    WeatherAgent,
    tools=[get_geolocation, get_current_weather],
    max_iters=6
)

## Uruchomienie agenta

Wywołujemy agenta jak zwykłą funkcję - argumenty muszą odpowiadać polom `InputField` z sygnatury, a wynik jest obiektem z polami zdefiniowanymi jako `OutputField`.

In [ ]:
result = weather_agent(user_question="Opisz jaka jest pogoda w Poznaniu. Czy powinienem wychodzić na spacer w stroju plażowym i okularach przeciwsłonecznych?")
print(result.answer)

## Trajektoria agenta - wszystkie kroki Thought / Action / Observation

`dspy.inspect_history` pokazuje pełną historię wymiany z modelem - widać tu po kolei rozumowanie modelu, wybrane akcje (wywołania funkcji) oraz wyniki, jakie agent dostał z powrotem.

In [ ]:
dspy.inspect_history(n=10)